# ADA 2026-B · Proyecto Oráculo (ADAG1)
## Notebook de Evaluación Oficial — Primera Entrega

**Grupo:** GP01 (ADAG1)  
**Integrantes:** Julián Gómez Ortiz, Juan Manoel  
**Semana:** 3  

---

### Instrucciones para el Docente
Este notebook está preparado para **ejecución directa y desatendida** sin necesidad de modificar variables, rutas ni parámetros de código:
1. Ejecutar las celdas en orden secuencial (`Entorno de ejecución` ▸ `Ejecutar todas`).
2. El notebook carga automáticamente la configuración ganadora almacenada en `entrega.json`.
3. Si el docente coloca su archivo de prueba privado (por ejemplo `datos_privados.json` o `test_privado.json`) en el entorno, el notebook lo detecta y prioriza automáticamente; de lo contrario, evalúa sobre el conjunto de validación curado de `datos_visibles.json`.
4. El resultado final se imprime de forma formateada con la precisión alcanzada y el desglose de restricciones.

### 1. Preparación del Entorno y Dependencias

In [ ]:
import sys
import os

# Si se ejecuta en Google Colab, instalar y configurar dependencias requeridas
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Configurando entorno Colab...")
    !pip install -q -U "bitsandbytes>=0.46.1" transformers accelerate nltk spacy emoji langdetect immutabledict
    !python -m spacy download en_core_web_sm -q
    if not os.path.exists("/content/open-instruct"):
        !git clone -q https://github.com/allenai/open-instruct /content/open-instruct
    
    # Descarga de tokenizadores de NLTK sin depender de endpoints bloqueados
    import io, urllib.request, zipfile
    NLTK_DATA = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages"
    for carp, nom in [("tokenizers", "punkt"), ("tokenizers", "punkt_tab"), ("taggers", "averaged_perceptron_tagger"), ("taggers", "averaged_perceptron_tagger_eng")]:
        try:
            with urllib.request.urlopen(f"{NLTK_DATA}/{carp}/{nom}.zip") as resp:
                zipfile.ZipFile(io.BytesIO(resp.read())).extractall(f"/root/nltk_data/{carp}")
        except Exception as e:
            pass
    print("Dependencias listas en Colab.")
else:
    print("Ejecutando en entorno local/servidor.")

### 2. Carga del Modelo Base y Módulos del Proyecto

In [ ]:
from ayudas import cargar_modelo, cargar_datos, dividir
from oraculo import Oraculo, CATALOGO

# Se carga el modelo oficial de referencia (qwen17b)
modelo = cargar_modelo("qwen17b")

### 3. Carga Automática de `entrega.json`

In [ ]:
import json

ruta_entrega = "entrega.json"
if not os.path.exists(ruta_entrega):
    # Intentar búsqueda en subdirectorios si Colab fue abierto desde la raíz
    posibles = [os.path.join(r, "entrega.json") for r, d, f in os.walk(".") if "entrega.json" in f]
    if posibles:
        ruta_entrega = posibles[0]

with open(ruta_entrega, "r", encoding="utf-8") as f:
    datos_entrega = json.load(f)

grupo = datos_entrega.get("grupo", "GP01")
semana = datos_entrega.get("semana", 3)
config = datos_entrega["config"]

print("=" * 60)
print(f"INFORMACIÓN DE ENTREGA CARGADA:")
print(f"  Grupo:        {grupo}")
print(f"  Semana:       {semana}")
print(f"  Configuración: {config}")
print("=" * 60)

# Detalle textual de los consejos seleccionados en el catálogo
print("\nDirectivas de prompt asociadas a la configuración:")
for ranura in ["rol", "estrategia", "formato", "estilo", "cierre"]:
    idx = config.get(ranura)
    texto = CATALOGO[ranura][idx] if idx is not None else "(ninguno)"
    print(f"- {ranura.upper()} (idx {idx}):\n  \"{texto}\"")

### 4. Detección y Preparación de Datos de Evaluación
El notebook revisa si existen datos privados del docente (`datos_privados.json`, `test_privado.json`, `evaluacion.json`). En caso contrario, emplea las instancias de prueba reservadas en `datos_visibles.json`.

In [ ]:
# Rutas potenciales para datos privados de evaluación del docente
candidatos_datos = [
    "datos_privados.json",
    "test_privado.json",
    "datos_evaluacion.json",
    "test.json",
    "datos_visibles.json"
]

ruta_eval = None
for c in candidatos_datos:
    if os.path.exists(c):
        ruta_eval = c
        break

if ruta_eval is None:
    # Buscar en árbol de trabajo
    for r, d, f in os.walk("."):
        for c in candidatos_datos:
            if c in f:
                ruta_eval = os.path.join(r, c)
                break
        if ruta_eval:
            break

print(f"Cargando archivo de datos: {ruta_eval}")
datos_crudos = cargar_datos(ruta_eval)

if "datos_visibles" in ruta_eval:
    busqueda, validacion = dividir(datos_crudos)
    datos_test = validacion
    print(f"Evaluando sobre partición de validación reservada ({len(datos_test)} instancias).")
else:
    # Si es archivo privado proporcionado por el profesor, se evalúan todas sus instancias
    datos_test = datos_crudos
    print(f"Evaluando sobre conjunto privado del docente ({len(datos_test)} instancias).")

# Inicialización del oráculo para evaluación
oraculo = Oraculo(modelo, datos=datos_test, validacion=datos_test, lote=16, presupuesto=32_000_000)

### 5. Ejecución de la Evaluación Oficial

In [ ]:
from collections import Counter

print("\nIniciando evaluación con el oráculo...")
resultado = oraculo.evaluar(config, datos_test, semilla=1)

correctas = resultado.n - len(resultado.trazas)

print("\n" + "=" * 60)
print("           RESULTADO DE LA EVALUACIÓN OFICIAL")
print("=" * 60)
print(f" Grupo evaluado:            {grupo}")
print(f" Total de instancias:       {resultado.n}")
print(f" Instancias cumplidas:      {correctas}")
print(f" Instancias falladas:       {len(resultado.trazas)}")
print(f" PRECISIÓN OBTENIDA:        {resultado.precision:.2%}")
print("=" * 60)

if resultado.trazas:
    print("\nTop 5 restricciones más frecuentemente incumplidas:")
    conteo_fallos = Counter(t["violo"] for t in resultado.trazas)
    for iid, frec in conteo_fallos.most_common(5):
        pct = (frec / resultado.n) * 100
        print(f"  - {iid}: {frec} fallos ({pct:.1f}% del total)")

### 6. Inspección de Muestra de Trazas (Opcional)
Muestra las primeras respuestas evaluadas y el motivo específico de rechazo por parte del verificador determinista.

In [ ]:
if resultado.trazas:
    print("Muestra de las primeras 2 trazas de fallo:\n")
    for i, t in enumerate(resultado.trazas[:2], 1):
        print(f"--- Fallo #{i} (ID Instancia: {t['id']}) ---")
        print(f"Restricción violada: {t['violo']}")
        salida_corta = t['salida'][:250].replace('\n', ' ')
        print(f"Salida del modelo:   \"{salida_corta}...\"\n")
else:
    print("¡Todas las instancias evaluadas cumplieron el 100% de las restricciones!")